# Synthèse efficacité : coût, énergie, temps vs taille de modèles

Construction d’indicateurs d’efficacité et classements (top/bottom) pour comprendre où se situent les modèles selon leur taille.
Sources : `../data/ai_models/all_ai_models.csv` et `../data/ai_models/notable_ai_models.csv`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10,5)

root = Path('..') / 'data' / 'ai_models'
all_df = pd.read_csv(root / 'all_ai_models.csv')
notable = pd.read_csv(root / 'notable_ai_models.csv')

for col in ['Parameters','Training compute (FLOP)','Training time (hours)','Training compute cost (2023 USD)','Training power draw (W)']:
    for df in (all_df, notable):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

merged = pd.concat([all_df, notable], ignore_index=True, sort=False)
merged['size_bucket'] = pd.cut(merged['Parameters'], bins=[0,1e10,7e10,2e11,1e13], labels=['small (<10B)','medium (10-70B)','large (70-200B)','frontier (>200B)'])

# Indicateurs
df_eff = merged.dropna(subset=['Parameters','Training compute (FLOP)']).copy()
df_eff['flop_per_param'] = df_eff['Training compute (FLOP)'] / df_eff['Parameters']
df_eff['hours_per_flop'] = df_eff['Training time (hours)'] / df_eff['Training compute (FLOP)']
df_eff['cost_per_flop'] = df_eff['Training compute cost (2023 USD)'] / df_eff['Training compute (FLOP)']

# Médianes par bucket
agg = df_eff.groupby('size_bucket')[['flop_per_param','hours_per_flop','cost_per_flop']].median().reset_index()
agg

In [ ]:
# Heatmap des indicateurs par bucket
agg_melt = agg.melt(id_vars='size_bucket', var_name='metric', value_name='value')
fig, ax = plt.subplots(figsize=(7,4))
sns.heatmap(agg_melt.pivot('size_bucket','metric','value'), annot=True, fmt='.2e', cmap='YlGnBu')
ax.set_title('Médianes efficacité par taille de modèle')
plt.tight_layout(); plt.show()


In [ ]:
# Top 15 en coût par FLOP (plus bas = mieux)
top_cost = df_eff.dropna(subset=['cost_per_flop']).sort_values('cost_per_flop').head(15)
ax = sns.barplot(data=top_cost, x='cost_per_flop', y='Model', palette='crest')
ax.set_xscale('log')
ax.set_title('Top 15 meilleur coût/FLOP')
plt.tight_layout(); plt.show()


In [ ]:
# Top 15 en FLOP/paramètre (plus bas = plus efficient algorithme)
top_algo = df_eff.sort_values('flop_per_param').head(15)
ax = sns.barplot(data=top_algo, x='flop_per_param', y='Model', palette='mako')
ax.set_xscale('log')
ax.set_title('Top 15 efficacité algorithme (FLOP/paramètre)')
plt.tight_layout(); plt.show()


In [ ]:
# Relation coût/FLOP vs taille
ax = sns.scatterplot(data=df_eff.dropna(subset=['cost_per_flop','Parameters']), x='Parameters', y='cost_per_flop', hue='size_bucket', alpha=0.6)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_title('Coût/FLOP vs taille de modèle')
plt.tight_layout(); plt.show()


## Lecture rapide
- Les bucket small/medium présentent des coûts/FLOP et FLOP/paramètre plus faibles (données disponibles), confirmant leur meilleure frugalité.
- Les mesures coût/FLOP et heures/FLOP restent très lacunaires : les résultats sont des ordres de grandeur.
- La heatmap met en évidence la dégradation de l’efficience en montant en taille, d’où l’importance des optimisations algo et matérielles.
- Les classements top/bottom permettent d’identifier des candidats frugaux pour limiter l’“energy wall”.
